# Golden Dataset Generation — Kaggle 2xT4

Generates ~100 QA pairs from documentation chunks using RAGAS.
These become the **golden dataset** used in `eval_pipeline.ipynb`.

## Overall flow
```
chunks_*.json
      │
      ├─ [Step 2] Filter junk + stratified sample → 250 parent chunks
      │           Convert to LangChain Documents
      │
      ├─ [Step 3] Install Ollama binary (GitHub Releases) + start server
      │           Pull Qwen2.5-32B Q4_K_M onto both T4 GPUs
      │
      ├─ [Step 4] Connect RAGAS to Ollama via instructor
      │           → LiteLLMStructuredLLM (InstructorBaseRagasLLM)
      │           → guaranteed-JSON code path, zero retry cascade
      │
      └─ [Step 5] Build KnowledgeGraph with CHUNK nodes manually
                    apply default_transforms_for_prechunked (skips HeadlineSplitter)
                    Part A — Knowledge Graph transforms (Qwen 32B via Ollama)
                              SummaryExtractor, NodeFilter,
                              ThemesExtractor, NERExtractor,
                              CosineSimilarity + OverlapScore edges
                    Part B — QA Synthesis via generator.generate()
                              SingleHop, MultiHopAbstract, MultiHopSpecific
                    Output → golden_dataset.json
```

## Why `generate_with_langchain_docs` + `transforms_llm` was abandoned

`generate_with_langchain_docs` creates `DOCUMENT` nodes internally.
`default_transforms_for_prechunked` only processes `CHUNK` nodes.
Passing `default_transforms_for_prechunked(...)` as `transforms_llm` is wrong —
`transforms_llm` expects a `BaseRagasLLM`, not a `Transforms` object.
RAGAS silently receives the wrong type and falls back to `default_transforms`,
which runs `HeadlinesExtractor` + `HeadlineSplitter` and creates `CHUNK` nodes
only from `DOCUMENT` nodes that have `>500 tokens`. With `MIN_LENGTH=500 chars`
(~125 tokens), almost no documents cross the token threshold, so `SummaryExtractor`
finds 0 eligible nodes — producing the '240 nodes have no summary' warnings.

**Fix:** build the `KnowledgeGraph` with `CHUNK` nodes ourselves, run
`apply_transforms` directly, then call `generator.generate()` — bypassing
`generate_with_langchain_docs` entirely.

## Why `LangchainLLMWrapper` was abandoned

Every previous crash (`fix_output_format`, `RagasOutputParserException`, kernel OOM)
came from one root cause: `LangchainLLMWrapper` puts RAGAS in a path where
it calls `parse_output_string` with a hardcoded 3-retry cascade via `fix_output_format`.
`RunConfig(max_retries=0)` does NOT affect this — it is hardcoded.
With a local 32B model returning plain text ~20% of the time, this creates an
exponential retry explosion that kills the kernel.

**Fix:** `LiteLLMStructuredLLM` (`InstructorBaseRagasLLM`) + Ollama constrained decoding
guarantees valid Pydantic objects on every call — no parsing, no retries, no cascade.

## GPU layout
| | Model | VRAM |
|---|---|---|
| Both T4s (30 GB total) | Qwen2.5-32B Q4_K_M via Ollama | ~20 GB — Ollama auto-splits |
| CPU | BAAI/bge-m3 embeddings | ~2 GB RAM |

## How to attach the chunks file on Kaggle
Kaggle notebook → **Add data** (top right) → Upload `chunks_*.json`.
It mounts at `/kaggle/input/<dataset-name>/`. Update `CHUNKS_FILE` in Step 2.

**Expected runtime: ~3–4 hrs for 250 chunks / 100 QA pairs with 32B model**

## Step 0 — Verify 2xT4 GPUs

In [1]:
import os, subprocess

# Prevent PyTorch CUDA memory fragmentation
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=index,name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
assert result.returncode == 0, "No GPU. Enable: Kaggle Settings → Accelerator → GPU T4 x2"

gpus = [g.strip() for g in result.stdout.strip().split('\n')]
print(f"Found {len(gpus)} GPU(s):")
for g in gpus: print(" ", g)

assert len(gpus) >= 2, f"Need 2 GPUs, found {len(gpus)}. Change accelerator to 'GPU T4 x2'."

Found 2 GPU(s):
  0, Tesla T4, 15360 MiB
  1, Tesla T4, 15360 MiB


## Step 1 — Install Dependencies

- `ragas==0.4.3` — pinned; `LiteLLMStructuredLLM` and `InstructorBaseRagasLLM` exist in this exact version
- `langchain-huggingface` + `sentence-transformers` — for the BGE-M3 embedding model
- `rapidfuzz` — RAGAS's `OverlapScoreBuilder` uses this for fuzzy named-entity matching between chunks;
  without it, overlap edges in the knowledge graph are silently skipped
- `openai` and `instructor` — already on Kaggle; verified below

In [2]:
!pip install -q "ragas==0.4.3" langchain-huggingface sentence-transformers rapidfuzz
import openai, instructor
print(f"openai {openai.__version__} | instructor {instructor.__version__}")
print("✅ Done.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 10.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 50.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 53.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 37.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 79.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are install

## Step 1b — Fix RAGAS Import Bug

RAGAS 0.4.3 has a hardcoded import of a VertexAI path that was removed from
`langchain-community` in recent versions:
```python
from langchain_community.chat_models.vertexai import ChatVertexAI  # no longer exists
```
We inject an empty stub module into `sys.modules` before any ragas import.
Python checks `sys.modules` before touching disk — the stub satisfies the import
and ragas loads cleanly. **This cell must run before every other ragas import.**

In [3]:
import sys, types

# Inject empty stub modules so RAGAS's broken legacy imports don't crash
for mod_path in ["langchain_community.chat_models.vertexai",
                 "langchain_community.llms.vertexai"]:
    stub = types.ModuleType(mod_path)
    stub.ChatVertexAI = type("ChatVertexAI", (), {})  # empty placeholder class
    stub.VertexAI     = type("VertexAI",     (), {})  # empty placeholder class
    sys.modules[mod_path] = stub

import ragas
print(f"✅ RAGAS {ragas.__version__} imported successfully.")

✅ RAGAS 0.4.3 imported successfully.


## Step 2 — Load & Prepare Documents

We filter low-quality parent chunks and apply stratified sampling across 4 frameworks,
then convert to LangChain `Document` objects.

**Why 250 chunks for 100 QA pairs?**
RAGAS's `CustomNodeFilter` drops ~60% of chunks as not QA-worthy (chunks that are
too generic or pure code with no prose). 250 → ~100 surviving nodes → ~100 QA pairs.
Set `TARGET=10` for a quick test run, `250` for the final run.

**Why sqrt-proportional sampling?**
langchain has 4,643 chunks vs langgraph's 264. Straight proportional sampling gives
langgraph only ~9 slots — too few for meaningful coverage. `sqrt(4643)≈68` vs
`sqrt(264)≈16` compresses the gap while still weighting by corpus size.

**Why `MIN_LENGTH=200` chars?**
We filter by character length here — `default_transforms_for_prechunked` does not
apply any token-based length filter, so all CHUNK nodes we create will be processed.
200 chars is the minimum for a chunk to support a meaningful question.

In [4]:
import json

# Update this path to match your Kaggle dataset mount
CHUNKS_FILE = "/kaggle/input/datasets/rushanksavant/chunks-json/chunks_c728061_20260717.json"

with open(CHUNKS_FILE) as f:
    all_chunks = json.load(f)

print(f"Loaded {len(all_chunks)} total chunks.")

Loaded 28006 total chunks.


In [5]:
import math, random
from collections import Counter
from langchain_core.documents import Document

random.seed(42)
TARGET     = 250     # ← change to 250 for the final run
JUNK       = ['TODO', '<CardGroup', '<div ', '{/*']
MIN_LENGTH = 1500    # chars — minimum for a chunk to support a meaningful QA pair

# Keep only parent chunks with real documentation content.
# Child chunks are small retrieval fragments — only parents have full section text.
def is_good(chunk):
    if chunk['type'] != 'parent': return False
    if len(chunk['text']) < MIN_LENGTH: return False
    if any(m in chunk['text'] for m in JUNK): return False  # MDX boilerplate / TODOs
    return True

quality = [c for c in all_chunks if is_good(c)]
print(f"Quality parents: {len(quality)}")
print("Per framework:", dict(Counter(c['metadata']['framework'] for c in quality)))

# Group by framework for stratified sampling
by_fw = {}
for c in quality:
    by_fw.setdefault(c['metadata']['framework'], []).append(c)

# Sqrt-proportional weighting: compresses the large/small framework gap
sqrt_w  = {fw: math.sqrt(len(v)) for fw, v in by_fw.items()}
total_w = sum(sqrt_w.values())
slots   = {fw: max(1, round(TARGET * w / total_w)) for fw, w in sqrt_w.items()}

print("\nSampling slots:")
for fw, n in slots.items():
    print(f"  {fw:12s}: {n} (from {len(by_fw[fw])} available)")

sampled = []
for fw, n in slots.items():
    sampled.extend(random.sample(by_fw[fw], min(n, len(by_fw[fw]))))

# Convert to LangChain Documents.
# page_content = the text RAGAS generates questions from.
# metadata is passed through to the output dataset for analysis.
documents = [
    Document(
        page_content = c['text'],
        metadata     = {
            'framework'  : c['metadata']['framework'],
            'source_file': c['metadata']['source_file'],
            'title'      : c['metadata'].get('global_title', ''),
            'section'    : c['metadata'].get('section_heading', ''),
            'parent_id'  : c['id'],
        }
    )
    for c in sampled
]

print(f"\nTotal sampled: {len(sampled)} chunks → {len(documents)} LangChain Documents")

Quality parents: 2073
Per framework: {'langchain': 1008, 'langsmith': 796, 'deepagents': 139, 'langgraph': 130}

Sampling slots:
  langchain   : 95 (from 1008 available)
  langsmith   : 85 (from 796 available)
  deepagents  : 35 (from 139 available)
  langgraph   : 34 (from 130 available)

Total sampled: 249 chunks → 249 LangChain Documents


## Step 3 — Install & Start Ollama

Ollama is a local model server that:
- Downloads and manages model weights
- Loads them across **both T4 GPUs automatically** — no manual GPU config needed
- Exposes an **OpenAI-compatible REST API** at `localhost:11434`

We download the Ollama binary directly from **GitHub Releases** instead of the
official install script (`ollama.com/install.sh`) because Kaggle's network policy
blocks outbound requests to `ollama.com`. GitHub Releases are on the allowlist.

**Model: `qwen2.5:32b-instruct-q4_K_M`** (~20 GB)
- 32B parameters → higher quality summaries, NER, and QA than 14B
- Q4_K_M: mixed 4/5-bit quantization, best quality-size tradeoff
- ~20 GB fits across 2×15 GB T4s with activation headroom
- 32B Q4 outperforms 14B Q8 — more parameters > more precision beyond a threshold

**Expected time: ~10–15 min** (binary ~1 min + model download ~10–12 min)

In [6]:
# import subprocess, os

# # Try official install script first
# result = subprocess.run(
#     "curl -fsSL https://ollama.com/install.sh | sh",
#     shell=True, capture_output=True, text=True
# )

# if result.returncode == 0:
#     print("✅ Installed via install.sh")
# else:
#     print("install.sh blocked, falling back to GitHub Releases archive ...")
    
#     # Install zstd for extraction
#     subprocess.run(["apt-get", "install", "-y", "-q", "zstd"], check=True)
    
#     # Download the full archive
#     subprocess.run([
#         "curl", "-fsSL",
#         "https://github.com/ollama/ollama/releases/latest/download/ollama-linux-amd64.tar.zst",
#         "-o", "/tmp/ollama.tar.zst"
#     ], check=True)
    
#     # Extract to /usr — the archive is structured as usr/bin/ollama + usr/lib/ollama/*
#     # Extracting to / places everything at the correct absolute paths
#     subprocess.run(
#         ["tar", "-I", "zstd", "-xf", "/tmp/ollama.tar.zst", "-C", "/"],
#         check=True
#     )
#     print("✅ Extracted full Ollama archive to /")

# result = subprocess.run(["ollama", "--version"], capture_output=True, text=True)
# print(f"✅ Ollama: {result.stdout.strip()}")



import subprocess, os, shutil

# Set Ollama concurrency limits BEFORE starting the server process
os.environ["OLLAMA_NUM_PARALLEL"] = "1"
os.environ["OLLAMA_MAX_LOADED_MODELS"] = "1"
os.environ["OLLAMA_KEEP_ALIVE"] = "24h"

print("1. Installing system dependency (zstd)...")
subprocess.run("apt-get update -qq && apt-get install -y -qq zstd", shell=True, check=True)

print("2. Cleaning up any lingering executable...")
if os.path.exists("/usr/local/bin/ollama"):
    os.remove("/usr/local/bin/ollama")

print("3. Downloading and installing Ollama...")
# Run install script without check=True so Docker/systemd errors don't stop execution
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True)

# 4. Verify the executable binary actually exists now
if shutil.which("ollama") or os.path.exists("/usr/local/bin/ollama"):
    print("✅ Ollama binary successfully installed!")
else:
    raise RuntimeError("Ollama binary failed to extract. Check logs above.")



# Reliable Kaggle-compatible install
# import subprocess, os

# OLLAMA_VERSION = "v0.5.13"  # Pin a specific version
# TARBALL = f"https://github.com/ollama/ollama/releases/download/{OLLAMA_VERSION}/ollama-linux-amd64.tgz"

# subprocess.run(["curl", "-fsSL", TARBALL, "-o", "/tmp/ollama.tgz"], check=True)
# subprocess.run(["tar", "-xzf", "/tmp/ollama.tgz", "-C", "/usr/local/"], check=True)
# os.chmod("/usr/local/bin/ollama", 0o755)

# result = subprocess.run(["ollama", "--version"], capture_output=True, text=True, check=True)
# print(f"✅ {result.stdout.strip()}")

1. Installing system dependency (zstd)...


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


Selecting previously unselected package zstd.
(Reading database ... 125186 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
2. Cleaning up any lingering executable...
3. Downloading and installing Ollama...


>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%##############   99.1%
>>> Creating ollama user...
>>> Adding ollama user to video group...


✅ Ollama binary successfully installed!


>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [7]:
import subprocess, time, urllib.request

MODEL = "qwen2.5:32b-instruct-q4_K_M"

# Start Ollama server in background.
# subprocess.Popen (not subprocess.run) — 'ollama serve' is a long-running server process,
# it must run in the background while the rest of the notebook executes.
# Logs go to /tmp/ollama.log so they don't flood the notebook output.
with open("/tmp/ollama.log", "w") as log_file:
    ollama_server_process = subprocess.Popen(
        ["ollama", "serve"],
        stdout=log_file,
        stderr=log_file,
    )
    time.sleep(5)

# Block until the server is ready — retry every 3 seconds for up to 90 seconds.
# A bare time.sleep(3) is unreliable; the model pull would fail if the server isn't up yet.
print("Waiting for Ollama server to start ...")
for attempt in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434", timeout=2)
        print("✅ Ollama server ready.")
        break
    except Exception:
        time.sleep(3)
else:
    raise RuntimeError("Ollama did not start after 90s. Check /tmp/ollama.log")

# Pull the model — do NOT suppress stdout/stderr so download progress and errors are visible
print(f"\nPulling {MODEL} (~20 GB) — this takes ~10–12 min ...")
subprocess.run(["ollama", "pull", MODEL], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Confirm it's available
models = subprocess.run(["ollama", "list"], capture_output=True, text=True)
print(models.stdout)
print(f"✅ Model {MODEL} ready.")




# import subprocess, time, urllib.request

# MODEL = "qwen2.5:32b-instruct-q4_K_M"  # Previously used qwen2.5:14b-instruct-q4_K_M

# # 1. Start Ollama server in background.
# # Popen (not subprocess.run) because 'ollama serve' never terminates — it keeps running as a server while the rest of the notebook executes.
# # Logs go to /tmp/ollama.log to avoid flooding the notebook output.
# with open("/tmp/ollama.log", "w") as log_file:
#     ollama_server_process = subprocess.Popen(
#         ["ollama", "serve"],
#         stdout=log_file,
#         stderr=log_file,
#     )

# # 2. Wait until the server is ready — retry every 3 seconds for up to 90 seconds.
# # the server may not be ready yet, and then the model pull below would fail silently.
# print("Waiting for Ollama server to start ...")
# for attempt in range(30):
#     try:
#         urllib.request.urlopen("http://127.0.0.1:11434", timeout=2)
#         print("✅ Ollama server ready.")
#         break
#     except Exception:
#         print("Ollama server not ready yet, trying again in 3-sec.")
#         time.sleep(3)
# else:
#     # If we exhaust all retries, raise — there is no point continuing
#     raise RuntimeError("Ollama did not start after 90s. Check /tmp/ollama.log")


# # 3. Pull model quietly (suppressing stderr/stdout progress updates)
# print(f"Pulling {MODEL} (~20.0 GB)... Please wait.")

# result = subprocess.run(
#     ["ollama", "pull", MODEL],
#     stdout=subprocess.DEVNULL,
#     stderr=subprocess.DEVNULL
# )

# if result.returncode == 0:
#     print(f"✅ Model {MODEL} downloaded successfully!")
# else:
#     print(f"❌ Failed to download model {MODEL}.")

Waiting for Ollama server to start ...
✅ Ollama server ready.

Pulling qwen2.5:32b-instruct-q4_K_M (~20 GB) — this takes ~10–12 min ...
NAME                           ID              SIZE     MODIFIED               
qwen2.5:32b-instruct-q4_K_M    9f13ba1299af    19 GB    Less than a second ago    

✅ Model qwen2.5:32b-instruct-q4_K_M ready.


## Step 4 — Connect RAGAS to Ollama via Instructor

### Why this eliminates all JSON parsing errors

RAGAS has two internal code paths based on LLM type:

**Path 1 — `LangchainLLMWrapper` (broken for local models):**
RAGAS gets a string back, tries to parse it as JSON via `parse_output_string`,
and on failure calls `fix_output_format` (another LLM call to repair the output).
This retry depth is hardcoded at 3 — `RunConfig(max_retries=0)` does not affect it.
Local models return plain text ~20% of the time → exponential retry cascade → kernel crash.

**Path 2 — `InstructorBaseRagasLLM` (what we use):**
RAGAS calls `llm.agenerate(prompt, response_model=SomePydanticSchema)` directly.
`instructor` enforces the schema via Ollama's constrained decoding — the model
physically cannot output tokens that violate the schema. RAGAS receives a guaranteed
Pydantic object, calls `result.model_dump_json()`, done. No parsing, no retries.

### How instructor + Ollama enforces schemas
`Mode.JSON_SCHEMA` sends `response_format: {type: json_schema, json_schema: ...}`
to Ollama. Ollama (v0.5+) uses constrained decoding — at each token generation step,
tokens that would make the output invalid for the schema are zeroed out before sampling.

In [8]:
import instructor
from openai import AsyncOpenAI
from ragas.llms.litellm_llm import LiteLLMStructuredLLM

# Standard AsyncOpenAI client pointed at Ollama's local OpenAI-compatible API.
# api_key can be any string — Ollama does not authenticate.
ollama_async_client = AsyncOpenAI(
    base_url = "http://localhost:11434/v1",
    api_key  = "ollama",
)

# instructor patches the client to enforce Pydantic schemas on every call.
# Mode.JSON_SCHEMA = constrained decoding via response_format json_schema (Ollama >= 0.5.0)
instructor_client = instructor.from_openai(
    ollama_async_client,
    mode=instructor.Mode.JSON_SCHEMA,
)

# LiteLLMStructuredLLM is InstructorBaseRagasLLM — RAGAS's guaranteed-JSON path.
# _check_client_async() checks client.__class__.__name__ == 'AsyncInstructor' → True
# so is_async=True and RAGAS calls agenerate() not generate().
ragas_llm = LiteLLMStructuredLLM(
    client   = instructor_client,
    model    = MODEL,
    provider = "openai",   # Ollama speaks the OpenAI protocol
)

print(f"LLM type : {type(ragas_llm).__name__}")
print(f"Is async : {ragas_llm.is_async}")
print(f"Model    : {ragas_llm.model}")

assert ragas_llm.is_async, "Expected AsyncInstructor — check instructor version"
print("✅ RAGAS LLM ready. Guaranteed-JSON path active.")

LLM type : LiteLLMStructuredLLM
Is async : True
Model    : qwen2.5:32b-instruct-q4_K_M
✅ RAGAS LLM ready. Guaranteed-JSON path active.


In [9]:
from pydantic import BaseModel

# Smoke test: verify instructor + Ollama enforces schema before the long generation run.
# If this fails, the whole pipeline will fail — better to know now.
class TestSchema(BaseModel):
    score: int
    reason: str

result = await ragas_llm.agenerate(
    prompt         = "Rate LangChain documentation quality 1-5 with a brief reason.",
    response_model = TestSchema,
)

print("Response type :", type(result).__name__)
print("Response value:", result)
assert isinstance(result, TestSchema), "Schema enforcement failed!"
print("✅ Structured output verified — model returned exact schema.")

Response type : TestSchema
Response value: score=4 reason="LangChain's documentation is quite comprehensive and useful for developers looking to integrate language models into their applications, but there might be room for improvement in clarity or additional examples."
✅ Structured output verified — model returned exact schema.


In [10]:
from langchain_huggingface import HuggingFaceEmbeddings

# BAAI/bge-m3: 8192 token limit — covers all parent chunks (P95 ≈ 1333 tokens).
# all-MiniLM-L6-v2 has a 512 token limit — would silently truncate long chunks,
# producing wrong cosine similarity scores and bad knowledge graph edges.
# Runs on CPU so it doesn't compete with Ollama for GPU VRAM.
print("Loading BAAI/bge-m3 embeddings on CPU ...")
embedding_model = HuggingFaceEmbeddings(
    model_name   = "BAAI/bge-m3",
    model_kwargs = {'device': 'cpu'},
)

test_emb = embedding_model.embed_query("test")
print(f"✅ Embedding model ready. Dim: {len(test_emb)}")

Loading BAAI/bge-m3 embeddings on CPU ...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

✅ Embedding model ready. Dim: 1024


## Step 5 — Generate the Dataset

### Why we build the KnowledgeGraph manually (not via `generate_with_langchain_docs`)

`generate_with_langchain_docs` always creates `DOCUMENT` nodes internally.
`default_transforms_for_prechunked` only processes `NodeType.CHUNK` nodes.
These two are incompatible when called through `generate_with_langchain_docs`.

The solution: build the `KnowledgeGraph` ourselves with `CHUNK` nodes,
run `apply_transforms` directly, then call `generator.generate()`.
This gives us the prechunked pipeline without any node-type mismatch.

### What `default_transforms_for_prechunked` runs (Part A)
For each CHUNK node, the 32B model produces structured outputs via constrained decoding:
- `SummaryExtractor` → `{"text": "..."}` — summary of the chunk
- `CustomNodeFilter` → `{"score": N}` — QA-worthiness; low-score chunks are dropped
- `ThemesExtractor` → `{"themes": [...]}` — key topics
- `NERExtractor` → `{"entities": [...]}` — class/method/framework names

BGE-M3 then embeds summaries (CPU) and RAGAS builds graph edges:
- `CosineSimilarityBuilder` — connects chunks with similar summaries
- `OverlapScoreBuilder` — connects chunks sharing named entities

### What `generator.generate()` runs (Part B)
- `SingleHopSpecific` (40%) — Q from one chunk, tests direct retrieval
- `MultiHopAbstract` (30%) — conceptual Q spanning two related chunks
- `MultiHopSpecific` (30%) — specific Q requiring two chunks, tests sub-query decomposition

In [11]:
import warnings
from transformers import logging as hf_logging

# Suppress noisy but harmless warnings from transformers internals
warnings.filterwarnings("ignore", message=r".*max_new_tokens.*max_length.*")
warnings.filterwarnings("ignore", message=r".*AttentionMaskConverter.*")
hf_logging.set_verbosity_error()

In [12]:
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.testset import TestsetGenerator
from ragas.testset.synthesizers import (
    SingleHopSpecificQuerySynthesizer,
    MultiHopAbstractQuerySynthesizer,
    MultiHopSpecificQuerySynthesizer,
)
from ragas.run_config import RunConfig

# Domain context injected into every synthesizer prompt.
# Tells the model what types of questions are in scope vs out of scope.
# Our agent has a guardrail that rejects API-reference questions — so we exclude them here too.
DOMAIN_CONTEXT = (
    "This corpus covers official documentation for LangChain, LangGraph, "
    "LangSmith, and the DeepAgents framework. "
    "Generate questions about concepts, usage patterns, configuration, and integration. "
    "DO NOT generate questions about specific class signatures, constructor parameters, "
    "method return types, or API reference content — those are out of scope."
)

# Wrap the HuggingFace embedding model for RAGAS internal use
ragas_emb = LangchainEmbeddingsWrapper(embedding_model)

# Instantiate TestsetGenerator directly — NOT via from_langchain().
# from_langchain() always wraps the llm in LangchainLLMWrapper internally,
# which reverts RAGAS to the parse_output_string retry cascade path.
# Direct instantiation preserves ragas_llm as LiteLLMStructuredLLM.
generator = TestsetGenerator(
    llm             = ragas_llm,
    embedding_model = ragas_emb,
    llm_context     = DOMAIN_CONTEXT,
)

# Each synthesizer also receives ragas_llm directly (guaranteed-JSON path)
query_distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=ragas_llm, llm_context=DOMAIN_CONTEXT), 0.60),
    (MultiHopAbstractQuerySynthesizer(llm=ragas_llm,  llm_context=DOMAIN_CONTEXT), 0.40),
    # (MultiHopSpecificQuerySynthesizer(llm=ragas_llm,  llm_context=DOMAIN_CONTEXT), 0.30),
]

# max_retries=3: retries cover network/timeout failures, not JSON parse failures
#   (instructor guarantees valid JSON so parse retries never trigger)
# timeout=600: 10 min per call — 32B model can take 60-120s on long chunks
# max_workers=1: one concurrent call at a time — prevents KV cache OOM on 32B
run_config = RunConfig(max_retries=3, timeout=600, max_workers=1)

print("✅ Generator, query distribution, and run config ready.")

✅ Generator, query distribution, and run config ready.


/tmp/ipykernel_58/3618165007.py:22: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_emb = LangchainEmbeddingsWrapper(embedding_model)


In [13]:
from ragas.testset.transforms.extractors.llm_based import SummaryExtractor, NERExtractor, ThemesExtractor
from ragas.testset.transforms.filters import CustomNodeFilter
from ragas.testset.transforms.extractors.embeddings import EmbeddingExtractor
from ragas.testset.transforms.relationship_builders.cosine import CosineSimilarityBuilder
from ragas.testset.transforms.relationship_builders.traditional import OverlapScoreBuilder
from ragas.testset.transforms import Parallel
from ragas.testset.graph import KnowledgeGraph, Node, NodeType
from ragas.testset.transforms import default_transforms_for_prechunked, default_transforms, apply_transforms

# ── Part A: Build Knowledge Graph ─────────────────────────────────────────
#
# We build the KG manually with NodeType.CHUNK nodes.
# generate_with_langchain_docs creates NodeType.DOCUMENT nodes internally,
# which are incompatible with default_transforms_for_prechunked (which only
# processes CHUNK nodes). Building the KG ourselves bypasses this mismatch.

kg = KnowledgeGraph()
for doc in documents:
    kg.nodes.append(Node(
        # type       = NodeType.CHUNK,
        type       = NodeType.DOCUMENT,
        properties = {
            "page_content"      : doc.page_content,
            "document_metadata" : doc.metadata,
        }
    ))

print(f"KnowledgeGraph initialised: {len(kg.nodes)} CHUNK nodes")
print(f"Model    : {MODEL}")
print(f"LLM type : {type(ragas_llm).__name__} (InstructorBaseRagasLLM — guaranteed JSON)")
print("\nRunning Part A — Knowledge Graph transforms ...")
print("  SummaryExtractor → CustomNodeFilter → ThemesExtractor + NERExtractor → similarity edges")

def filter_docs(node):
    return node.type == NodeType.DOCUMENT

transforms = [
    SummaryExtractor(llm=ragas_llm, filter_nodes=filter_docs),
    CustomNodeFilter(llm=ragas_llm, filter_nodes=filter_docs),
    Parallel(
        EmbeddingExtractor(
            embedding_model    = ragas_emb,
            property_name      = "summary_embedding",
            embed_property_name= "summary",
            filter_nodes       = filter_docs,
        ),
        ThemesExtractor(llm=ragas_llm, filter_nodes=filter_docs),
        NERExtractor(llm=ragas_llm,    filter_nodes=filter_docs),
    ),
    Parallel(
        CosineSimilarityBuilder(
            property_name     = "summary_embedding",
            new_property_name = "summary_similarity",
            threshold         = 0.65,
            filter_nodes      = filter_docs,
        ),
        OverlapScoreBuilder(threshold=0.01, filter_nodes=filter_docs),
    ),
]

apply_transforms(kg, transforms, run_config)

# Count nodes that have summaries (successfully processed)
nodes_with_summary = sum(1 for n in kg.nodes if n.properties.get("summary"))
print(f"\nPart A complete.")
print(f"  Nodes with summary : {nodes_with_summary} / {len(kg.nodes)}")
print(f"  Relationships built: {len(kg.relationships)}")

doc_nodes   = [n for n in kg.nodes if n.type == NodeType.DOCUMENT]
chunk_nodes = [n for n in kg.nodes if n.type == NodeType.CHUNK]
child_rels  = [r for r in kg.relationships if r.type == "child"]




# Assign the completed KG to the generator
generator.knowledge_graph = kg

# Diagnostic — run this before generator.generate()
print(f"KG nodes      : {len(generator.knowledge_graph.nodes)}")
print(f"KG relationships: {len(generator.knowledge_graph.relationships)}")

# Check relationship types
rel_types = {}
for r in generator.knowledge_graph.relationships:
    rel_types[r.type] = rel_types.get(r.type, 0) + 1
print(f"Relationship types: {rel_types}")

# Check if summary_similarity property exists on any relationship
ss_rels = [r for r in generator.knowledge_graph.relationships
           if r.get_property("summary_similarity") is not None]
print(f"Rels with summary_similarity property: {len(ss_rels)}")

KnowledgeGraph initialised: 249 CHUNK nodes
Model    : qwen2.5:32b-instruct-q4_K_M
LLM type : LiteLLMStructuredLLM (InstructorBaseRagasLLM — guaranteed JSON)

Running Part A — Knowledge Graph transforms ...
  SummaryExtractor → CustomNodeFilter → ThemesExtractor + NERExtractor → similarity edges


Applying SummaryExtractor:   0%|          | 0/249 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/249 [00:00<?, ?it/s]

Applying EmbeddingExtractor:   0%|          | 0/249 [00:00<?, ?it/s]

Applying ThemesExtractor:   0%|          | 0/249 [00:00<?, ?it/s]

Applying NERExtractor:   0%|          | 0/249 [00:00<?, ?it/s]

Applying CosineSimilarityBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]


Part A complete.
  Nodes with summary : 249 / 249
  Relationships built: 1144
KG nodes      : 249
KG relationships: 1144
Relationship types: {'summary_similarity': 819, 'entities_overlap': 325}
Rels with summary_similarity property: 819


In [14]:
# ── Part B: QA Synthesis ──────────────────────────────────────────────────
#
# generator.generate() uses the KG built in Part A to synthesize QA pairs.
# It picks node clusters (single nodes or connected pairs) and asks the
# 32B model to generate a question + ground-truth answer grounded in those nodes.

print("\nRunning Part B — QA synthesis ...")
testset = generator.generate(
    testset_size       = 150,          # ← change to 100 for the final run
    query_distribution = query_distribution,
    run_config         = run_config,
    raise_exceptions   = False,       # skip individual failures, don't crash the run
)

print(f"\n✅ Generation complete. Got {len(testset)} samples.")


Running Part B — QA synthesis ...


Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/147 [00:00<?, ?it/s]


✅ Generation complete. Got 147 samples.


## Step 6 — Validate & Save

Even with constrained decoding guaranteeing valid JSON schema, the **semantic content**
can still be poor. If a chunk passes `CustomNodeFilter` but the LLM still can't
form a grounded question (e.g. pure code chunk with no prose), the generated answer
may contain fallback phrases. We filter those rows out.

The output `golden_dataset.json` contains:
- `user_input` — the question fed to the agent during evaluation
- `reference` — ground-truth answer for scoring
- `reference_contexts` — source chunk texts (used for `ContextRecall` metric)
- `question_type` — for per-type score breakdowns in `eval_pipeline.ipynb`

In [19]:
import pandas as pd
pd.set_option('display.max_colwidth', 120)

df = testset.to_pandas()
print(f"Columns: {df.columns.tolist()}")
print(f"Rows   : {len(df)}")
display(df[['user_input', 'reference', 'synthesizer_name']].head(5))

Columns: ['user_input', 'reference_contexts', 'reference', 'persona_name', 'query_style', 'query_length', 'synthesizer_name']
Rows   : 147


,user_input,reference,synthesizer_name
0,Whate ar langchain integrtions with Azure?,langchain integrates with Azure through two main options for accessing embedding models: \n\n1. **Azure OpenAI** (re...,single_hop_specific_query_synthesizer
1,wat is bert in SAP HANA Cloud data graph,Bert is a Puppet that appears in the Sesame Street show and is stored as data within the Puppets graph in SAP HANA C...,single_hop_specific_query_synthesizer
2,"As a Tech Contributor to the langchain framework, how should I use GitHub Issues when reporting bugs?","When reporting bugs for the langchain framework, you should first check if the issue already exists in the GitHub Is...",single_hop_specific_query_synthesizer
3,What benefits does Amazon AWS Lambda provide for developers in the langchain framework?,Amazon AWS Lambda is a serverless computing service provided by Amazon Web Services (AWS) that helps developers buil...,single_hop_specific_query_synthesizer
4,How can openai:gpt-5.5 be integrated using LangChain in a web application?,openai:gpt-5.5 can be integrated into a web application using the LangChain framework by creating an agent with spec...,single_hop_specific_query_synthesizer


In [20]:
# Drop rows where RAGAS couldn't generate a grounded question.
# Happens for pure-code chunks that passed NodeFilter but have no prose to QA on.
BAD = [
    "insufficient information",
    "i don't know",
    "cannot be determined",
    "not mentioned in the context",
]

def is_valid(row):
    q = str(row.get('user_input', ''))
    a = str(row.get('reference', ''))
    if len(q.strip()) < 15 or len(a.strip()) < 30: return False
    if any(p in a.lower() for p in BAD): return False
    return True

df_valid = df[df.apply(is_valid, axis=1)].reset_index(drop=True)
print(f"Before filter: {len(df)} | After filter: {len(df_valid)}")

if len(df_valid) < 50:
    print("⚠️ Fewer than 50 valid pairs. Increase TARGET in Step 2.")

Before filter: 147 | After filter: 147


In [21]:
import json

# /kaggle/working/ is the only writable directory on Kaggle.
# Files saved here appear in the notebook Output tab and can be downloaded.
OUTPUT = "/kaggle/working/golden_dataset.json"

golden_dataset = [
    {
        "user_input"         : str(row['user_input']).strip(),
        "reference"          : str(row['reference']).strip(),
        "reference_contexts" : row.get('reference_contexts', []),
        "question_type"      : str(row.get('synthesizer_name', '')).strip(),
    }
    for _, row in df_valid.iterrows()
]

with open(OUTPUT, 'w') as f:
    json.dump(golden_dataset, f, indent=2, ensure_ascii=False)

print(f"✅ Saved {len(golden_dataset)} QA pairs to {OUTPUT}")
print("   Download: Kaggle notebook → Output tab → golden_dataset.json")

if golden_dataset:
    s = golden_dataset[0]
    print(f"\nSample entry:")
    print(f"  Q   : {s['user_input']}")
    print(f"  A   : {s['reference'][:150]} ...")
    print(f"  Type: {s['question_type']}")

✅ Saved 147 QA pairs to /kaggle/working/golden_dataset.json
   Download: Kaggle notebook → Output tab → golden_dataset.json

Sample entry:
  Q   : Whate ar langchain integrtions with Azure?
  A   : langchain integrates with Azure through two main options for accessing embedding models: 

1. **Azure OpenAI** (recommended) — Use embedding models de ...
  Type: single_hop_specific_query_synthesizer


In [22]:
# Save the knowledge graph
kg.save("/kaggle/working/knowledge_graph.json")

In [23]:
# Shut down Ollama server cleanly — releases GPU VRAM
ollama_server_process.terminate()
print("✅ Ollama server stopped.")

✅ Ollama server stopped.
